# Does the irregular grid break downstream QC tools?

`test_irregular_grid_boustrophedon_return_path.ipynb` confirmed the
single-axis-adaptive irregular grid ("Method 2") + `determine_return_side`
don't contradict `acquisition/positions.py`'s own boustrophedon-path/
return-path builders -- its own Takeaways call this "safe to promote... not
done in this notebook, per the request to test first."

Before actually promoting anything, an exhaustive downstream audit (every
module/notebook that reads `positions_*.txt` or FOV positions) found the
core pipeline is safe (it only ever consumes FOV counts or raw (x, y) pairs)
but turned up three real, *unverified* places that assume a regular
Cartesian lattice:

1. **Per-FOV heatmaps** (`after_imaging/04_view_intensity_stats.ipynb`,
   `during_imaging/stage_z_drift.ipynb`, and two more copies) reshape FOV
   positions into a dense `(n_y, n_x)` matrix by ranking unique x/y values --
   an irregular grid's cross-axis values differ per row/column, so this
   won't produce a small dense matrix the way it does today.
2. **FFC exterior-FOV selection** (`analysis/ffc.py`'s default
   `"exterior_grid"` strategy) calls `find_exterior_fovs` with one nominal
   `config.step_size_um` -- an irregular grid's real local spacing on the
   adaptive axis differs from that constant.
3. **Camera-rotation neighbour-finding**
   (`notebooks/misc/correct_camera_rotation.ipynb`'s `find_3x3_block`)
   needs a complete 8-connected 3x3 FOV neighbourhood and raises
   `RuntimeError` if none exists.

This notebook measures all three effects on the same real boundary the
prior two notebooks used, and prototypes/tests a fix for (1) and (2) --
(3) is checked honestly for whether a fix is even possible, not forced.
Nothing here changes `acquisition/positions.py`, `analysis/ffc.py`, or any
`before_imaging`/`after_imaging`/`during_imaging` notebook -- purely a
diagnostic, feeding evidence into the still-pending production plan.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.geometry import box as shapely_box
from shapely.ops import unary_union
from scipy.spatial import cKDTree

# notebooks/tests/<subfolder>/ is three levels under the repo root (MERci/),
# same convention as notebooks/before_imaging/regular/ (3 levels).
MERCI_DIR  = Path(os.getcwd()).parent.parent.parent   # MERci/
SAMPLE_DIR = MERCI_DIR.parent                  # this repo's own sandbox experiment
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.acquisition.configs   import get_camera_pixel_size_um, get_camera_frame_size
from MERci.visualization import get_merci_figures_dir
from MERci.acquisition.positions import (
    load_boundary_polygon, load_hole_polygons,
    filter_scanning_path, get_path_stats,
    optimize_grid_offset, find_exterior_fovs, find_grid_neighbor,
    spaced_coords,   # production internal -- reused, not reimplemented
)

FIG_DIR = get_merci_figures_dir(SAMPLE_DIR, "tests", "test_irregular_grid_downstream_qc_tools", subfolder="irregular_grid")
FIG_DIR.mkdir(parents=True, exist_ok=True)

## Part 0 -- rebuild the real irregular grid (same boundary as the prior two notebooks)

`build_irregular_bands`/`generate_irregular_scanning_path`/
`determine_return_side` are duplicated here unchanged from
`test_irregular_grid_boustrophedon_return_path.ipynb` (same convention that
notebook already used, reusing production internals rather than
reimplementing them) -- this notebook's own contribution starts at Part 1.
`fixed_axis="x"` is used throughout, matching
`compare_60x_40x_irregular_grid_fov_coverage.ipynb`'s own selected winner
(fewest FOVs on this real tissue).

In [ ]:
def build_irregular_bands(boundary_polygon, hole_polygons, step_size_um, fov_size_um,
                           fixed_axis="y", min_width_frac=0.1, fixed_offset=0.0):
    tissue = boundary_polygon.difference(unary_union(hole_polygons)) if hole_polygons else boundary_polygon
    xmin, ymin, xmax, ymax = boundary_polygon.bounds
    half_h = fov_size_um / 2.0
    min_width_um = min_width_frac * fov_size_um

    if fixed_axis == "y":
        fixed_min, fixed_max, cross_min, cross_max = ymin, ymax, xmin, xmax
    elif fixed_axis == "x":
        fixed_min, fixed_max, cross_min, cross_max = xmin, xmax, ymin, ymax
    else:
        raise ValueError("fixed_axis must be 'x' or 'y'")

    fixed_center = (fixed_min + fixed_max) / 2.0 + fixed_offset
    fixed_positions = spaced_coords(fixed_center, fixed_min, fixed_max, step_size_um, even=True)

    bands = []
    for f in fixed_positions:
        lo_f, hi_f = f - half_h, f + half_h
        strip = (shapely_box(cross_min - 1.0, lo_f, cross_max + 1.0, hi_f) if fixed_axis == "y"
                 else shapely_box(lo_f, cross_min - 1.0, hi_f, cross_max + 1.0))
        inter = tissue.intersection(strip)
        cross_vals = []
        if not inter.is_empty:
            pieces = list(inter.geoms) if hasattr(inter, "geoms") else [inter]
            pieces.sort(key=lambda p: p.bounds[0] if fixed_axis == "y" else p.bounds[1])
            for piece in pieces:
                if piece.is_empty:
                    continue
                pxmin, pymin, pxmax, pymax = piece.bounds
                lo, hi = (pxmin, pxmax) if fixed_axis == "y" else (pymin, pymax)
                if (hi - lo) < min_width_um:
                    continue
                piece_positions = spaced_coords((lo + hi) / 2.0, lo, hi, step_size_um, even=False)
                cross_vals.extend(piece_positions.tolist())
        bands.append((float(f), np.array(sorted(cross_vals))))
    return bands


def generate_irregular_scanning_path(bands, fixed_axis):
    path = []
    n_bands = len(bands)
    if fixed_axis == "y":
        for strip, i in enumerate(range(n_bands - 1, -1, -1)):
            fixed_val, cross_vals = bands[i]
            ordered = cross_vals if strip % 2 == 0 else cross_vals[::-1]
            for c in ordered:
                path.append((c, fixed_val))
    elif fixed_axis == "x":
        for j in range(n_bands):
            fixed_val, cross_vals = bands[j]
            ordered = cross_vals[::-1] if j % 2 == 0 else cross_vals
            for c in ordered:
                path.append((fixed_val, c))
    else:
        raise ValueError("fixed_axis must be 'x' or 'y'")
    return np.array(path) if path else np.empty((0, 2))


BOUNDARY_DIR = MERCI_DIR / "data" / "positions" / "examples" / "lineage_tracing_mosaic"
boundary_polygon = load_boundary_polygon(BOUNDARY_DIR / "boundary_positions.txt")
hole_polygons    = load_hole_polygons(BOUNDARY_DIR)

MICROSCOPE = "ST2"
image_size_px, _  = get_camera_frame_size(MICROSCOPE)
pixel_size_60x_um = get_camera_pixel_size_um(MICROSCOPE)
pixel_size_40x_um = pixel_size_60x_um * (60.0 / 40.0)
NON_OVERLAP_FRACTION = 0.9
fov_size_um  = pixel_size_40x_um * image_size_px
step_size_um = fov_size_um * NON_OVERLAP_FRACTION
FIXED_AXIS   = "x"

bands = build_irregular_bands(boundary_polygon, hole_polygons, step_size_um, fov_size_um, fixed_axis=FIXED_AXIS)
irregular_path = generate_irregular_scanning_path(bands, fixed_axis=FIXED_AXIS)
irregular_coords = filter_scanning_path(irregular_path, boundary_polygon, hole_polygons, fov_size_um)

regular_result = optimize_grid_offset(boundary_polygon, hole_polygons, step_size_um, fov_size_um,
                                       direction="vertical", n_samples=9)
regular_coords = regular_result.coords

print(f"irregular grid (fixed_axis='{FIXED_AXIS}'): {len(irregular_coords)} FOVs")
print(f"regular grid (offset-optimised):             {len(regular_coords)} FOVs")

irregular_positions = {i: tuple(c) for i, c in enumerate(irregular_coords)}
regular_positions   = {i: tuple(c) for i, c in enumerate(regular_coords)}

## Part 1 -- per-FOV heatmap reshaping

`_positions_to_grid_indices`/`plot_fov_heatmap`
(`after_imaging/04_view_intensity_stats.ipynb`, duplicated near-verbatim in
3 more places) ranks unique rounded x/y values into a dense `(n_y, n_x)`
matrix. Reproduced verbatim below on synthetic per-FOV intensities, for
both grids, to measure -- not assume -- how bad the irregular case actually
is.

In [ ]:
def positions_to_grid_indices(fov_ids, positions):
    """Verbatim port of after_imaging/04_view_intensity_stats.ipynb's
    _positions_to_grid_indices, generalised to a plain {fov_id: (x, y)}
    dict instead of ExperimentMetadata (same rounding/ranking logic)."""
    xs = np.array([round(positions[f][0]) for f in fov_ids])
    ys = np.array([round(positions[f][1]) for f in fov_ids])
    unique_xs = np.sort(np.unique(xs))
    unique_ys = np.sort(np.unique(ys))
    x_rank = {v: i for i, v in enumerate(unique_xs)}
    y_rank = {v: i for i, v in enumerate(unique_ys)}
    return {f: (x_rank[xs[i]], y_rank[ys[i]]) for i, f in enumerate(fov_ids)}


def heatmap_matrix(positions, rng_seed=0):
    """Synthetic 'intensity' (smooth spatial trend + noise) mapped into
    the dense (n_y, n_x) matrix plot_fov_heatmap builds today -- returns
    (matrix, nan_fraction, n_x, n_y) so the degradation is a number, not a
    visual impression."""
    fov_ids = sorted(positions.keys())
    rng = np.random.default_rng(rng_seed)
    xs = np.array([positions[f][0] for f in fov_ids])
    ys = np.array([positions[f][1] for f in fov_ids])
    intensity = 500 + 0.02 * (xs - xs.mean()) + rng.normal(0, 15, size=len(fov_ids))

    grid = positions_to_grid_indices(fov_ids, positions)
    n_x = max(xi for xi, _ in grid.values()) + 1
    n_y = max(yi for _, yi in grid.values()) + 1
    matrix = np.full((n_y, n_x), np.nan)
    for k, f in enumerate(fov_ids):
        xi, yi = grid[f]
        matrix[yi, xi] = intensity[k]
    nan_fraction = float(np.isnan(matrix).sum() / matrix.size)
    return matrix, nan_fraction, n_x, n_y


mat_reg, nan_reg, nx_reg, ny_reg = heatmap_matrix(regular_positions)
mat_irr, nan_irr, nx_irr, ny_irr = heatmap_matrix(irregular_positions)

print(f"regular grid:   matrix shape {mat_reg.shape} ({nx_reg}x{ny_reg}) for {len(regular_positions)} FOVs, "
      f"{nan_reg:.1%} NaN")
print(f"irregular grid: matrix shape {mat_irr.shape} ({nx_irr}x{ny_irr}) for {len(irregular_positions)} FOVs, "
      f"{nan_irr:.1%} NaN")

### Fix: guard on the matrix-size blowup factor, fall back to a scatter plot otherwise

A real regular grid is never a perfectly dense rectangle either (tissue
shape/holes leave real gaps -- confirmed above: the regular grid's own
`unique_x * unique_y` already exceeds `n_fovs` by a modest ~1.4x on this real
tissue), so exact Cartesian-product equality is the wrong test. What
actually distinguishes the irregular grid is the SIZE of that blowup: 1.4x
for the regular grid vs. tens of times for the irregular grid (measured
above), because the adaptive axis's per-band re-centring turns what would
be one shared column/row into hundreds of near-duplicate ones. Guard on a
blowup-factor threshold (`unique_x * unique_y > 3 * n_fovs`) instead, and
fall back to a colour-scatter render when it fires. Verifies the
regular-grid case is untouched (still `imshow`) and the irregular case
renders something legible instead of a near-empty, oversized matrix.

In [ ]:
BLOWUP_THRESHOLD = 3.0   # unique_x * unique_y / n_fovs above this -> not a usable dense grid

def plot_fov_heatmap_guarded(positions, rng_seed=0, title=""):
    fov_ids = sorted(positions.keys())
    rng = np.random.default_rng(rng_seed)
    xs = np.array([positions[f][0] for f in fov_ids])
    ys = np.array([positions[f][1] for f in fov_ids])
    intensity = 500 + 0.02 * (xs - xs.mean()) + rng.normal(0, 15, size=len(fov_ids))

    unique_x, unique_y = np.unique(np.round(xs)), np.unique(np.round(ys))
    blowup = len(unique_x) * len(unique_y) / len(fov_ids)
    is_cartesian = blowup <= BLOWUP_THRESHOLD

    fig, ax = plt.subplots(figsize=(6, 5))
    if is_cartesian:
        grid = positions_to_grid_indices(fov_ids, positions)
        n_x = max(xi for xi, _ in grid.values()) + 1
        n_y = max(yi for _, yi in grid.values()) + 1
        matrix = np.full((n_y, n_x), np.nan)
        for k, f in enumerate(fov_ids):
            xi, yi = grid[f]
            matrix[yi, xi] = intensity[k]
        im = ax.imshow(matrix, cmap="viridis", origin="upper", interpolation="nearest")
        plt.colorbar(im, ax=ax, fraction=0.035, pad=0.04, label="intensity")
        mode = f"imshow (blowup={blowup:.2f}x <= {BLOWUP_THRESHOLD}x)"
    else:
        sc = ax.scatter(xs, ys, c=intensity, cmap="viridis", s=18)
        ax.set_aspect("equal")
        ax.invert_yaxis()
        plt.colorbar(sc, ax=ax, fraction=0.035, pad=0.04, label="intensity")
        mode = f"scatter fallback (blowup={blowup:.2f}x > {BLOWUP_THRESHOLD}x)"

    ax.set_title(f"{title}\n{mode}", fontsize=10)
    fig.tight_layout()
    return fig, is_cartesian


fig_reg, cartesian_reg = plot_fov_heatmap_guarded(regular_positions, title=f"regular grid ({len(regular_positions)} FOVs)")
fig_reg.savefig(FIG_DIR / "test_irregular_grid_downstream_qc_heatmap_regular.png", dpi=150, bbox_inches="tight")
plt.show()

fig_irr, cartesian_irr = plot_fov_heatmap_guarded(irregular_positions, title=f"irregular grid ({len(irregular_positions)} FOVs)")
fig_irr.savefig(FIG_DIR / "test_irregular_grid_downstream_qc_heatmap_irregular.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"regular grid detected as Cartesian:   {cartesian_reg} (expected True)")
print(f"irregular grid detected as Cartesian: {cartesian_irr} (expected False -- triggers the fallback)")
assert cartesian_reg, "regular grid should always be detected as a clean Cartesian product."
assert not cartesian_irr, "irregular grid unexpectedly looked Cartesian on this real tissue -- re-check."
print("\nPASS: the Cartesian-product guard fires exactly where the plain dense-matrix heatmap breaks down, "
      "and leaves the regular-grid case unaffected.")

## Part 2 -- FFC exterior-FOV selection

`analysis/ffc.py`'s default `"exterior_grid"` strategy calls
`find_exterior_fovs(positions, config.step_size_um, tolerance_fraction=
config.ffc_neighbor_tolerance)` -- one nominal step size for the whole
grid. Compared here against a geometry-only ground truth that doesn't
assume any step size at all: a FOV is exterior if it has no real neighbour
roughly towards each of the 4 cardinal directions within a generous search
radius.

In [ ]:
def geometric_exterior_ground_truth(positions, fov_size_um, angle_tol_deg=30.0, max_radius_factor=1.75):
    """Step-size-free ground truth: a FOV is exterior if it has no real
    neighbour within a generous radius roughly towards each cardinal
    direction (angular sector match), independent of any assumed lattice."""
    fov_ids = sorted(positions.keys())
    coords = np.array([positions[f] for f in fov_ids])
    tree = cKDTree(coords)
    max_r = max_radius_factor * fov_size_um
    directions = {"E": 0.0, "N": 90.0, "W": 180.0, "S": 270.0}

    exterior = set()
    for i, (x, y) in enumerate(coords):
        idxs = [j for j in tree.query_ball_point((x, y), r=max_r) if j != i]
        if not idxs:
            exterior.add(fov_ids[i])
            continue
        vecs = coords[idxs] - np.array([x, y])
        angles = np.degrees(np.arctan2(vecs[:, 1], vecs[:, 0])) % 360.0
        for base_angle in directions.values():
            diff = np.abs(((angles - base_angle + 180.0) % 360.0) - 180.0)
            if not np.any(diff <= angle_tol_deg):
                exterior.add(fov_ids[i])
                break
    return exterior


ground_truth = geometric_exterior_ground_truth(irregular_positions, fov_size_um)

nn_dists = cKDTree(irregular_coords).query(irregular_coords, k=2)[0][:, 1]
empirical_step_um = float(np.median(nn_dists))

exterior_nominal   = find_exterior_fovs(irregular_positions, step_size_um, tolerance_fraction=0.25)
exterior_empirical = find_exterior_fovs(irregular_positions, empirical_step_um, tolerance_fraction=0.25)

def _compare(name, predicted):
    false_pos = predicted - ground_truth
    false_neg = ground_truth - predicted
    print(f"{name:32s}: {len(predicted):4d} predicted exterior vs {len(ground_truth):4d} ground truth "
          f"-- false_pos={len(false_pos):3d}, false_neg={len(false_neg):3d}")
    return len(false_pos), len(false_neg)

print(f"nominal step_size_um={step_size_um:.1f}, empirical (median NN dist)={empirical_step_um:.1f}\n")
fp_nom,  fn_nom  = _compare("nominal config.step_size_um", exterior_nominal)
fp_emp,  fn_emp  = _compare("empirical median-NN step",    exterior_empirical)

Neither step size choice helps -- both give the identical, badly
over-predicting result (~4x too many "exterior" FOVs). That's expected in
hindsight: the within-band spacing IS exactly `step_size_um` by
construction, so the empirical median-NN estimate just measures the same
number back. The real problem isn't the step MAGNITUDE, it's that
`tolerance_fraction` (`config.ffc_neighbor_tolerance`, default 0.25) is too
tight to absorb the cross-band PHASE shift (different bands' cross-axis
lattices are independently re-centred, up to `step_size_um/2` apart) -- the
same mechanism Part 3 measures directly. Swept below, the one knob that
actually exists in production (`ffc_neighbor_tolerance`), against the same
geometric ground truth.

In [ ]:
tol_rows = []
for tol in (0.25, 0.5, 0.75, 1.0, 1.5, 2.0):
    predicted = find_exterior_fovs(irregular_positions, step_size_um, tolerance_fraction=tol)
    fp = len(predicted - ground_truth)
    fn = len(ground_truth - predicted)
    tol_rows.append({"tolerance_fraction": tol, "n_predicted": len(predicted),
                      "false_pos": fp, "false_neg": fn, "total_error": fp + fn})

df_tol = pd.DataFrame(tol_rows)
df_tol

Whichever `tolerance_fraction` minimises `total_error` against the
geometric ground truth is the most defensible setting for an irregular-grid
tissue -- decided here with the actual numbers above, not assumed. Note
this is a real trade-off, not a clean fix: false positives drop as
tolerance widens, but false negatives rise (a real cross-band gap starts
looking like a false neighbour match), and past some point (here,
`tolerance_fraction &gt;= 1.5`) the search radius exceeds half the FOV size
and everything falsely looks interior.

In [ ]:
best_row = df_tol.loc[df_tol["total_error"].idxmin()]
default_row = df_tol[df_tol["tolerance_fraction"] == 0.25].iloc[0]
if best_row["tolerance_fraction"] == 0.25:
    print("CONCLUSION: the current default ffc_neighbor_tolerance=0.25 is already the best setting tested "
          "on this irregular grid -- no change needed.")
else:
    print(f"CONCLUSION: raising ffc_neighbor_tolerance from the default 0.25 (total_error="
          f"{int(default_row['total_error'])}) to {best_row['tolerance_fraction']} "
          f"(total_error={int(best_row['total_error'])}) meaningfully reduces exterior-FOV "
          f"misclassification on this irregular grid -- but does not reach zero error at any setting "
          f"tested (see the table above): a real, quantified trade-off, not a full fix.")

## Part 3 -- camera-rotation 3x3 neighbourhoods

`notebooks/misc/correct_camera_rotation.ipynb`'s `find_3x3_block` needs a
complete 8-connected neighbourhood (`find_grid_neighbor` in all 4 cardinal
+ 4 diagonal directions) around some centre FOV. For `fixed_axis="x"`,
`"up"`/`"down"` neighbours are WITHIN one band (same x, adjacent y on the
same regularly-spaced cross lattice) -- reliable. `"left"`/`"right"`
neighbours are ACROSS bands (adjacent x, same y) -- each band's cross
lattice is independently re-centred, so an exact same-y match is not
generally expected. Checked directly below, not assumed, and swept over
`tolerance_fraction` to see whether widening it is a real fix or just
admits wrong matches.

In [ ]:
def find_3x3_block(fov_ids, positions, step_size_um, tolerance_fraction):
    """Verbatim port of notebooks/misc/correct_camera_rotation.ipynb's
    find_3x3_block, reusing production's find_grid_neighbor unchanged."""
    def _diag(fov_a, dir_from_a, fov_b, dir_from_b):
        d = find_grid_neighbor(fov_a, positions, dir_from_a, step_size_um, tolerance_fraction)
        if d is None:
            d = find_grid_neighbor(fov_b, positions, dir_from_b, step_size_um, tolerance_fraction)
        return d

    for center in fov_ids:
        up    = find_grid_neighbor(center, positions, "up",    step_size_um, tolerance_fraction)
        down  = find_grid_neighbor(center, positions, "down",  step_size_um, tolerance_fraction)
        left  = find_grid_neighbor(center, positions, "left",  step_size_um, tolerance_fraction)
        right = find_grid_neighbor(center, positions, "right", step_size_um, tolerance_fraction)
        if None in (up, down, left, right):
            continue
        up_left    = _diag(up, "left", left, "up")
        up_right   = _diag(up, "right", right, "up")
        down_left  = _diag(down, "left", left, "down")
        down_right = _diag(down, "right", right, "down")
        if None in (up_left, up_right, down_left, down_right):
            continue
        return center
    return None


fov_ids = sorted(irregular_positions.keys())
rows = []
for tol in (0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50):
    n_up_down    = sum(1 for f in fov_ids
                       if find_grid_neighbor(f, irregular_positions, "up", step_size_um, tol) is not None
                       and find_grid_neighbor(f, irregular_positions, "down", step_size_um, tol) is not None)
    n_left_right = sum(1 for f in fov_ids
                       if find_grid_neighbor(f, irregular_positions, "left", step_size_um, tol) is not None
                       and find_grid_neighbor(f, irregular_positions, "right", step_size_um, tol) is not None)
    block_center = find_3x3_block(fov_ids, irregular_positions, step_size_um, tol)
    rows.append({"tolerance_fraction": tol,
                 "n_with_up_down (within-band)": n_up_down,
                 "n_with_left_right (cross-band)": n_left_right,
                 "n_fovs": len(fov_ids),
                 "complete_3x3_block_found": block_center is not None})

df_3x3 = pd.DataFrame(rows)
df_3x3

An honest negative result here (no complete 3x3 block, or only at a
tolerance wide enough to risk matching the wrong neighbour) is an
acceptable conclusion, not a fix to force -- `correct_camera_rotation.ipynb`
would then be a known, documented limitation for irregular-grid tissues
rather than something patched blindly.

In [ ]:
any_found = df_3x3["complete_3x3_block_found"].any()
if not any_found:
    print("CONCLUSION: no complete 3x3 (8-connected) FOV neighbourhood exists at any tolerance tested "
          "(0.10-0.50) on this irregular grid -- correct_camera_rotation.ipynb cannot be used as-is on an "
          "irregular-grid tissue. This is a genuine, documented limitation, not something to patch here.")
else:
    first_ok = df_3x3[df_3x3["complete_3x3_block_found"]].iloc[0]
    print(f"CONCLUSION: a complete 3x3 block first appears at tolerance_fraction="
          f"{first_ok['tolerance_fraction']}. Whether that tolerance is still tight enough to trust for "
          f"rotation fitting (vs. wide enough to admit a wrong neighbour) needs a human judgement call, "
          f"not an automatic pass.")

## Takeaways

- **Heatmap reshaping (Part 1) -- real degradation, real fix**: measured, not
  assumed -- the irregular grid's per-FOV heatmap matrix blows up to a
  33x-oversized, 97%-NaN `(531, 36)` matrix vs. the regular grid's modest
  1.4x/30.6%-NaN `(22, 36)` (real tissue holes already make even the
  regular grid non-dense, so exact Cartesian-product equality is the wrong
  test -- the blowup FACTOR is what actually separates the two). A
  threshold guard (`unique_x * unique_y > 3 * n_fovs` -> fall back) fires
  exactly on the irregular case and leaves the regular case as plain
  `imshow`, unchanged. This fix is ready to port into the 4 notebooks that
  duplicate this helper.
- **FFC exterior-FOV selection (Part 2) -- degraded, no clean fix, but a
  real trade-off exists**: both the nominal `config.step_size_um` and an
  empirical median-NN-distance step give the IDENTICAL, badly over-
  predicting result (521 predicted vs. 126 geometric-ground-truth exterior
  FOVs) -- expected in hindsight, since the within-band spacing already
  equals `step_size_um` exactly by construction, so an empirical step just
  re-measures the same number. The real cause is `tolerance_fraction`
  (`config.ffc_neighbor_tolerance`, default 0.25) being too tight for the
  cross-band phase shift (Part 3's same root mechanism). Sweeping it
  (`0.25` to `2.0`) shows a real, quantified trade-off -- see the printed
  CONCLUSION line for the exact best setting on this tissue -- but no
  setting reaches zero error: false positives drop as tolerance widens,
  false negatives rise, and past `tolerance_fraction=1.5` the search radius
  exceeds half the FOV size and the count collapses to zero. Recommend
  widening `ffc_neighbor_tolerance` for irregular-grid tissues to the
  swept optimum rather than leaving the regular-grid default in place, but
  document the residual error rather than claiming a full fix.
- **Camera-rotation 3x3 neighbourhoods (Part 3)** -- see the printed
  CONCLUSION line above. The within-band (`up`/`down` for `fixed_axis="x"`)
  vs. cross-band (`left`/`right`) split in the swept table shows directly
  whether the failure (if any) is specifically the cross-band, independently
  -re-centred lattice -- exactly the mechanism the production audit
  predicted, now measured rather than assumed.
- Nothing in `acquisition/positions.py`, `analysis/ffc.py`, or any
  `before_imaging`/`after_imaging`/`during_imaging` notebook was changed by
  this notebook -- these three CONCLUSION lines are the evidence to act on
  in the next (still-pending) step, promoting the irregular grid into
  production.